# 06 — TabFM vs. trained models: next-week red tide risk

## Objective
Test Google's **TabFM** (a pre-trained "zero-shot" foundation model for tables) against the models the team already
uses, on the same task, data, split and scores, following the advisor's guidance from November 2026:

| Advisor priority | How this notebook applies it |
|---|---|
| The main product is a **regional** next-week risk forecast. Site-level prediction is secondary until it is shown to be reliable across sites | The regional weekly table (`model_table_weekly_fe_reduced.parquet`, the same file as `06_LightGBM_Boosting_Model.ipynb`) is the main benchmark. The site-week table (`site_week_table_fe.parquet`) is a secondary section |
| **Onset recall** is a primary metric, reported with missed blooms and false alarms | Every leaderboard leads with onset recall, missed blooms and false alarms, and ranks models on them |
| Missing a bloom costs more than a false alarm, about 2:1 or 3:1 | Two cost scores, `cost_2to1` and `cost_3to1`, shown next to the raw counts |
| Environmental variables should not be forced in if they reduce validation performance; document what was tested and what the model relies on | Section 5 tests each environmental group, section 6 shows what each model relies on |

> Predict next week's *K. brevis* class (`y_class`: **0** < 5,000, **1** = 5,000–99,999, **2** ≥ 100,000 cells/L)
> from what we know this week.

### Why the regional comparison is run year by year
The regional table has only **2 onset weeks in 2024 and none in 2025**, so one validation year cannot measure onset
recall. Section 4 predicts every year from 2017 to 2024 with models trained only on the years before it, which pools
**25 onsets**. Notebook 05 used the same approach.

### What TabFM is
TabFM is not trained on our data. `fit()` only stores the training rows; at prediction time the model reads them as
*context* (in-context learning, like a prompt for an LLM) and predicts the new rows in one forward pass. It was
pre-trained on hundreds of millions of synthetic tables. Google reports it beats tuned XGBoost / CatBoost / LightGBM on
TabArena (38 classification datasets of 700–150,000 rows). The regional table (364 training weeks) is smaller than
any of those, so this is a real test of whether the claim carries over to our data.

### What this notebook runs
| Section | Question |
|---|---|
| 3. Regional, 2024 | Train ≤ 2023 → predict 2024: the same split as the LightGBM notebook |
| 4. Regional, year by year | **Main comparison.** Which model catches the most onsets at an acceptable cost? |
| 5. Regional feature groups | Does any model use the environmental groups better? |
| 6. Regional drivers | Which features each model relies on |
| 7. Regional final check | Train ≤ 2024 → predict 2025 (once), and false alarms in 2026 |
| 8. Site-week (secondary) | The same comparison per sampling site, plus a learning curve |
| 9. Verdict | Printed automatically from the numbers above |

### How to run it in Colab
1. **Runtime → Change runtime type → T4 GPU** (or better).
2. **Runtime → Run all.** The first cell installs TabFM and CatBoost; the data is downloaded from the GitHub repo.
3. Expect roughly 20–40 minutes on a T4, most of it TabFM on the site-week table. Switches in the **Settings** cell
   turn sections off.

Within each table, every model gets the **same rows, features, split and decision rule**, so the leaderboards are
comparable.

## 0. Setup

In [ ]:
# Colab already has pandas, scikit-learn, PyTorch, LightGBM and XGBoost. This adds TabFM (PyTorch backend) and CatBoost.
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tabfm[pytorch]", "catboost"], check=True)
else:
    print('Not in Colab: install the packages yourself if needed: pip install "tabfm[pytorch]" catboost lightgbm xgboost')

In [ ]:
import inspect
import json
import os
import time
import traceback
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix,
                             roc_auc_score, average_precision_score, log_loss, roc_curve, precision_recall_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import xgboost as xgb

try:
    import catboost
except ImportError:
    catboost = None

try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch, HAS_GPU = None, False

try:
    import tabfm
    from tabfm import TabFMClassifier
    HAS_TABFM = hasattr(tabfm, "tabfm_v1_0_0_pytorch")
except ImportError:
    HAS_TABFM = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none (TabFM will be slow on CPU)")
print("TabFM installed:", HAS_TABFM, "| CatBoost installed:", catboost is not None)
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "| torch", torch.__version__ if torch else "-")

### Settings
Change these and re-run. `SCORE_TEST` controls the final checks in sections 7 and 8: the project rule is to look at
2025 only for the final comparison, so leave it off while you are still changing models or features.

In [ ]:
SEED = 42

FEATURE_SET = "core"            # "core", "core+temperature", "core+chlorophyll", "core+wind", "core+all"
DECISION_RULE = "balanced"      # "balanced" (divide by the training class shares, like class_weight="balanced") or "argmax"
COST_RATIOS = (2, 3)            # a missed bloom counts this many times as much as a false alarm
ROLLING_YEARS = list(range(2017, 2025))  # section 4: each year is predicted from all earlier years (2025 stays the test year)

RUN_TABFM = True                # TabFM, default settings (32 ensemble members)
RUN_TABFM_ENSEMBLE = True       # TabFM.ensemble(): feature crosses + SVD features + calibration (slower)
TABFM_BATCH_SIZE = 1            # ensemble members per forward pass; raise to 4-8 on a large GPU to go faster

RUN_REGIONAL_SWEEP = True       # section 5
RUN_REGIONAL_IMPORTANCE = True  # section 6 (trained models); TabFM importance is much slower:
RUN_TABFM_IMPORTANCE = False
SCORE_TEST = True               # sections 7 and 8: refit on <= 2024 and score 2025 + 2026

TUNE_TREES = True               # section 8: random search for LightGBM and XGBoost on the site-week 2024 year
N_TRIALS = 30                   # random-search trials per model
RUN_LEARNING_CURVE = True       # section 8

N_BOOT = 1000                   # bootstrap resamples for the confidence intervals
OUTPUT_DIR = "outputs_06"       # leaderboards are saved here as CSV
os.makedirs(OUTPUT_DIR, exist_ok=True)

if RUN_TABFM and not HAS_TABFM:
    print("TabFM is not installed, so the TabFM rows will be skipped.")

### Load the final modelling tables
The notebook looks for `data/final/` in the repo first (running locally), then in the team Drive folder, and otherwise
downloads the files from GitHub. `feature_sets.json` is stored with Git LFS, so it comes from the LFS download URL.

In [ ]:
REPO = "mhosigiri/Biointerphase-1B-forecasting-harmful-algal-bloom-risk"
BRANCH = "main"
FILES = ["site_week_table_fe.parquet", "model_table_weekly_fe_reduced.parquet", "feature_sets.json"]
RAW_URL = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/data/final/"
LFS_URL = f"https://media.githubusercontent.com/media/{REPO}/{BRANCH}/data/final/"


def is_lfs_pointer(path):
    with open(path, "rb") as f:
        return f.read(40).startswith(b"version https://git-lfs")


def find_data_dir():
    candidates = [os.path.join("..", "data", "final"), os.path.join("data", "final"),
                  "/content/drive/MyDrive/biointerphase_data/final"]
    for folder in candidates:
        if all(os.path.exists(os.path.join(folder, name)) and not is_lfs_pointer(os.path.join(folder, name))
               for name in FILES):
            return folder
    folder = "data_final"
    os.makedirs(folder, exist_ok=True)
    for name in FILES:
        path = os.path.join(folder, name)
        if not os.path.exists(path) or is_lfs_pointer(path):
            urllib.request.urlretrieve(RAW_URL + name, path)
            if is_lfs_pointer(path):
                urllib.request.urlretrieve(LFS_URL + name, path)
    return folder


DATA_DIR = find_data_dir()
site = pd.read_parquet(os.path.join(DATA_DIR, "site_week_table_fe.parquet"))
regional = pd.read_parquet(os.path.join(DATA_DIR, "model_table_weekly_fe_reduced.parquet"))
with open(os.path.join(DATA_DIR, "feature_sets.json")) as f:
    feature_sets = json.load(f)

print("Data folder:", os.path.abspath(DATA_DIR))
print("Site-week table:", site.shape, "| regional table:", regional.shape)

In [ ]:
# Keep labelled rows only: y_class is NaN when the next week was not sampled
site = site[site["y_class"].notna()].copy()
site["y_class"] = site["y_class"].astype(int)
regional = regional[regional["y_class"].notna()].copy()
regional["y_class"] = regional["y_class"].astype(int)

CLASSES = [0, 1, 2]
CLASS_NAMES = ["0 no bloom", "1 at risk", "2 bloom"]
SPLITS = ["train", "validate", "test", "holdout"]
TABLES = {"regional": (regional, "last_class"), "site-week": (site, "site_last_class")}  # table, last observed class


def is_onset(df, last_col):
    # A new bloom: the last observed class was 0 and next week reaches 5,000 cells/L
    return (df[last_col] == 0) & (df["y_class"] > 0)


for name, (df, last_col) in TABLES.items():
    counts = pd.crosstab(df["split"], df["y_class"]).reindex(index=SPLITS, columns=CLASSES, fill_value=0)
    counts.columns = CLASS_NAMES
    onsets = is_onset(df, last_col).groupby(df["split"]).sum().reindex(SPLITS, fill_value=0)
    display(Markdown(f"**{name} table**"))
    display(counts.assign(total=counts.sum(axis=1), onsets=onsets.astype(int)))

r_onsets = is_onset(regional, "last_class").groupby(regional["target_year"]).sum()
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(r_onsets.index.astype(str), r_onsets.values, color="#de2d26")
for i, v in enumerate(r_onsets.values):
    ax.text(i, v, f"{v}", ha="center", va="bottom", fontsize=8)
ax.set(title="Regional onset weeks per forecast year: too few in any one year to score onset recall", ylabel="onsets")
plt.tight_layout()
plt.show()

### Features
`regional_core` and `site_core` are the 12-feature sets chosen in notebook 05: bloom history, how long the current
state has lasted, and season (the site set adds the site's own and its neighbours' history). The candidate groups add
environmental features. Columns under `do_not_use` are never used.

In [ ]:
ENV_GROUPS = ["temperature", "chlorophyll", "wind"]   # what "core+all" adds


def feature_list(table, name):
    core = list(feature_sets[f"{table}_core"])
    groups = feature_sets[f"{table}_candidates"]
    if name == "core":
        return core
    if name == "core+all":
        return core + [c for g in ENV_GROUPS for c in groups[g]]
    return core + list(groups[name.split("+", 1)[1]])


R_FEATURES = feature_list("regional", FEATURE_SET)
FEATURES = feature_list("site", FEATURE_SET)
print(f"Feature set '{FEATURE_SET}': {len(R_FEATURES)} regional, {len(FEATURES)} site-week features")
print("Regional: ", R_FEATURES)
print("Site-week:", FEATURES)
display(regional.loc[regional["split"] == "train", R_FEATURES].isna().mean()
        .rename("missing share (regional train)").to_frame().T.style.format("{:.1%}"))

## 1. How every model is scored

Every model outputs three probabilities per row, and every score is computed from them the same way for every model.
The first rows are the advisor's priorities.

| Score | Meaning | Why |
|---|---|---|
| **Onset recall** | New blooms caught: the last observed class was 0, next week is ≥ 5k, and the model predicted 1 or 2 | Primary metric: the only weeks where copying last week cannot help |
| Onsets caught / onsets | The counts behind onset recall | The counts are small, so read the recall with them |
| **Missed blooms** | Real class 2 predicted as 0 | The costliest error |
| **False alarms** | Real class 0 predicted as 1 or 2 | Cost of extra sampling and warnings |
| **Cost 2:1 / 3:1** | 2 (or 3) × missed blooms + false alarms | A miss counts 2 (or 3) times as much as a false alarm. Lower is better |
| Macro-F1 | F1 averaged over the three classes | The project's main metric so far |
| Balanced accuracy, recall 1 / 2, precision 2 | Per-class detail | |
| AUC ≥5k, AP ≥5k | Ranking quality of P(class ≥ 1), independent of any threshold | |
| Onset AUC | AUC ≥5k on onset-eligible rows only | Can the model see a new bloom coming, at any threshold? |
| Log loss | Quality of the probabilities | Lower is better |

**Ranking.** `mean_rank` gives equal weight to three ranks: onset recall, cost (the 2:1 and 3:1 ranks averaged) and
macro-F1 (1 = best). The majority-class baseline never warns, so it is shown but not ranked.

**Reading the cost.** "No bloom" weeks far outnumber bloom weeks, so a model that warns more often adds many false
alarms for each miss it saves, and at 2:1 or 3:1 a model that rarely warns can have the lowest cost. That is why cost is
read together with onset recall, never on its own.

**Decision rule.** No model is trained with class weights. Instead, each model's probabilities are divided by the
class shares of its training rows before taking the most likely class (`DECISION_RULE = "balanced"`). This stands in
for `class_weight="balanced"` and applies equally to TabFM, which has no class-weight option. The cost scores
evaluate that decision; they are not used to choose it. The threshold-free scores (AUC, AP, log loss) do not depend on
it.

**Uncertainty.** Confidence intervals come from a bootstrap that resamples whole **weeks** (sites in the same week are
not independent). Differences inside the intervals are noise.

In [ ]:
def align_proba(classes, proba):
    # Put the probability columns in the order 0, 1, 2 (a class missing from training gets probability 0)
    proba = np.asarray(proba, dtype=float)
    out = np.zeros((len(proba), len(CLASSES)))
    for j, c in enumerate(np.asarray(classes).astype(int)):
        out[:, CLASSES.index(c)] = proba[:, j]
    return out / out.sum(1, keepdims=True).clip(1e-12)


def class_prior(y):
    return np.bincount(np.asarray(y, dtype=int), minlength=len(CLASSES)) / len(y)


def decide(proba, prior):
    # prior is one row of class shares, or one row per prediction when the rows come from different training sets
    if DECISION_RULE == "balanced":
        return np.argmax(proba / np.clip(prior, 1e-6, None), axis=1)
    return np.argmax(proba, axis=1)


def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan


COST_COLS = [f"cost_{r}to1" for r in COST_RATIOS]
COUNT_METRICS = ["onsets_caught", "onsets", "missed_blooms", "false_alarms"] + COST_COLS
METRICS = (["onset_recall"] + COUNT_METRICS +
           ["macro_f1", "balanced_acc", "recall_1", "recall_2", "precision_2", "auc_any", "ap_any", "onset_auc",
            "log_loss", "macro_f1_argmax"])
HIGHER_IS_BETTER = {"onset_recall": True, "onsets_caught": True, "onsets": True, "missed_blooms": False,
                    "false_alarms": False, **{c: False for c in COST_COLS}, "macro_f1": True, "balanced_acc": True,
                    "recall_1": True, "recall_2": True, "precision_2": True, "auc_any": True, "ap_any": True,
                    "onset_auc": True, "log_loss": False, "macro_f1_argmax": True}
RANK_GROUPS = {"onset recall": ["onset_recall"], "cost": COST_COLS, "macro-F1": ["macro_f1"]}  # equal weight each
BOOT_METRICS = ["onset_recall", COST_COLS[-1], "macro_f1", "auc_any"]


def metric_value(k, y, pred, proba, last):
    # One score for one model; y, pred, proba and last cover the same rows
    p_any = proba[:, 1] + proba[:, 2]
    onset_rows = last == 0
    onset = onset_rows & (y > 0)
    missed = int(((y == 2) & (pred == 0)).sum())
    false = int(((y == 0) & (pred > 0)).sum())
    if k == "onset_recall":
        return (pred[onset] > 0).mean() if onset.any() else np.nan
    if k == "onsets_caught":
        return int((pred[onset] > 0).sum())
    if k == "onsets":
        return int(onset.sum())
    if k == "missed_blooms":
        return missed
    if k == "false_alarms":
        return false
    if k.startswith("cost_"):
        return int(k[len("cost_"):-len("to1")]) * missed + false
    if k == "macro_f1":
        return f1_score(y, pred, average="macro", labels=CLASSES, zero_division=0)
    if k == "macro_f1_argmax":
        return f1_score(y, proba.argmax(axis=1), average="macro", labels=CLASSES, zero_division=0)
    if k == "balanced_acc":
        return balanced_accuracy_score(y, pred)
    if k in ("recall_1", "recall_2"):
        return recall_score(y, pred, labels=[int(k[-1])], average="macro", zero_division=0)
    if k == "precision_2":
        return precision_score(y, pred, labels=[2], average="macro", zero_division=0)
    if k == "auc_any":
        return safe_auc(y > 0, p_any)
    if k == "ap_any":
        return average_precision_score(y > 0, p_any) if (y > 0).any() else np.nan
    if k == "onset_auc":
        return safe_auc(y[onset_rows] > 0, p_any[onset_rows])
    if k == "log_loss":
        return log_loss(y, np.clip(proba, 1e-7, 1), labels=CLASSES)
    raise KeyError(k)


def score(y, proba, last_class, prior, metrics=METRICS):
    y = np.asarray(y, dtype=int)
    last = np.asarray(last_class, dtype=float)
    pred = decide(proba, prior)
    return {k: metric_value(k, y, pred, proba, last) for k in metrics}


def week_bootstrap(y, proba_by_model, weeks, last_class, prior, metrics, n_boot=N_BOOT):
    # Resample whole weeks with replacement; returns {metric: DataFrame(resample x model)}
    rng = np.random.default_rng(SEED)
    y = np.asarray(y, dtype=int)
    last = np.asarray(last_class, dtype=float)
    weeks = np.asarray(weeks)
    uniq = np.unique(weeks)
    rows_of = {w: np.flatnonzero(weeks == w) for w in uniq}
    preds = {m: decide(p, prior) for m, p in proba_by_model.items()}
    out = {k: [] for k in metrics}
    for _ in range(n_boot):
        idx = np.concatenate([rows_of[w] for w in rng.choice(uniq, len(uniq), replace=True)])
        for k in metrics:
            out[k].append({m: metric_value(k, y[idx], preds[m][idx], p[idx], last[idx])
                           for m, p in proba_by_model.items()})
    return {k: pd.DataFrame(v) for k, v in out.items()}

## 2. The models

| Model | Type | Settings |
|---|---|---|
| Majority class | baseline | always the training class shares |
| Persistence | baseline | next week = the last observed class (`last_class` regional, `site_last_class` site-week) |
| Logistic regression | linear | median imputation + missing flags, scaled, multinomial |
| Random forest | bagged trees | 500 trees, `min_samples_leaf=10` (as in `random_forest.ipynb`), NaN handled natively |
| HistGradientBoosting | boosted trees | scikit-learn defaults |
| LightGBM | boosted trees | regional: the settings tuned in `06_LightGBM_Boosting_Model.ipynb`; site-week: random search on 2024 (`TUNE_TREES`) |
| XGBoost | boosted trees | regional: fixed settings; site-week: random search on 2024 (`TUNE_TREES`) |
| CatBoost | boosted trees | 1,000 iterations, depth 6 |
| **TabFM** | foundation model | default: 32 ensemble members, all training rows as context |
| **TabFM ensemble** | foundation model | `TabFMClassifier.ensemble()`: feature crosses, SVD features, NNLS weights, calibration |

The LightGBM notebook weights classes with `class_weight="balanced"`. With that weighting, `LGB_REGIONAL` reproduces
its 2024 result exactly (macro-F1 0.717, 1 of 2 onsets caught, 3 false alarms), so both notebooks use the same rows and
features. Here the shared decision rule does the weighting for every model instead, so the LightGBM row differs from
that notebook's.

TabFM fills missing numbers with the column mean, so it also gets a 0/1 "was missing" column for every feature that
has gaps (the trees see the NaN directly). The README says the default context is 100 rows, but the code's default is
`max_num_rows=None`, which uses **all** training rows; it is set explicitly below.

In [ ]:
def with_missing_flags(X, cols):
    X = X.astype(float).copy()
    for c in cols:
        X[c + "_missing"] = X[c].isna().astype(float)
    return X


class Persistence:
    # Copies the last observed class; probability 1 on that class
    def __init__(self, column):
        self.column = column
        self.classes_ = np.array(CLASSES)

    def fit(self, X, y):
        return self

    def predict_proba(self, X):
        last = X[self.column].fillna(0).astype(int).values
        return np.eye(len(CLASSES))[last]


class Majority:
    def fit(self, X, y):
        self.classes_ = np.array(CLASSES)
        self.prior_ = class_prior(y)
        return self

    def predict_proba(self, X):
        return np.tile(self.prior_, (len(X), 1))


class TabFMWrapper:
    # TabFM with missing-value flags; one shared pre-trained model is reused by every instance
    _model = None

    def __init__(self, preset="default", n_estimators=32, cache_context=False):
        self.preset, self.n_estimators, self.cache_context = preset, n_estimators, cache_context

    @classmethod
    def load_model(cls):
        if cls._model is None:
            device = "cuda" if HAS_GPU else "cpu"
            # bfloat16 is the design default; T4-class GPUs (compute < 8.0) have no fast bf16, so use float32 there
            fast_bf16 = HAS_GPU and torch.cuda.get_device_capability(0)[0] >= 8
            dtype = torch.bfloat16 if fast_bf16 else torch.float32
            t0 = time.time()
            cls._model = tabfm.tabfm_v1_0_0_pytorch.load(model_type="classification", device=device, dtype=dtype)
            print(f"TabFM weights loaded on {device} ({dtype}) in {time.time() - t0:.0f} s")
        return cls._model

    def fit(self, X, y):
        model = self.load_model()
        self.flag_cols_ = [c for c in X.columns if X[c].isna().any()]
        kwargs = dict(max_num_rows=None, batch_size=TABFM_BATCH_SIZE, random_state=SEED)
        # cache_context is on TabFM's GitHub main but not in the PyPI release, so pass it only where it exists
        if self.cache_context and "cache_context" in inspect.signature(TabFMClassifier.__init__).parameters:
            kwargs["cache_context"] = True
        if self.preset == "ensemble":
            self.clf_ = TabFMClassifier.ensemble(model, **kwargs)
        else:
            self.clf_ = TabFMClassifier(model=model, n_estimators=self.n_estimators, **kwargs)
        self.clf_.fit(with_missing_flags(X, self.flag_cols_), np.asarray(y, dtype=int))
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, X):
        proba = np.asarray(self.clf_.predict_proba(with_missing_flags(X, self.flag_cols_)))
        if HAS_GPU:
            torch.cuda.empty_cache()
        return proba


def logistic():
    return make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                         LogisticRegression(C=1.0, max_iter=5000))


def random_forest():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=10, n_jobs=-1, random_state=SEED)


def hist_gb():
    return HistGradientBoostingClassifier(random_state=SEED)


def catboost_model():
    return catboost.CatBoostClassifier(iterations=1000, depth=6, loss_function="MultiClass", random_seed=SEED,
                                       verbose=0, allow_writing_files=False)


LGB_DEFAULT = dict(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=20, subsample=0.8,
                   subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0)
XGB_DEFAULT = dict(n_estimators=400, learning_rate=0.03, max_depth=4, min_child_weight=1, subsample=0.8,
                   colsample_bytree=0.8, reg_lambda=1.0)
# Tuned with time-aware cross-validation on the training years in 06_LightGBM_Boosting_Model.ipynb
LGB_REGIONAL = dict(n_estimators=100, learning_rate=0.03, max_depth=5, num_leaves=15)


def lightgbm_model(params):
    return lgb.LGBMClassifier(objective="multiclass", random_state=SEED, n_jobs=-1, verbose=-1, **params)


def xgboost_model(params):
    return xgb.XGBClassifier(objective="multi:softprob", tree_method="hist", device="cuda" if HAS_GPU else "cpu",
                             random_state=SEED, **params)

In [ ]:
BASELINES = ("Majority class", "Persistence")


def model_zoo(last_col, lgb_params, xgb_params, lgb_name="LightGBM", xgb_name="XGBoost", tabfm=True):
    # name -> a function that builds a fresh, unfitted model
    zoo = {
        "Majority class": lambda: Majority(),
        "Persistence": lambda: Persistence(last_col),
        "Logistic regression": logistic,
        "Random forest": random_forest,
        "HistGradientBoosting": hist_gb,
        lgb_name: lambda: lightgbm_model(lgb_params),
        xgb_name: lambda: xgboost_model(xgb_params),
    }
    if catboost is not None:
        zoo["CatBoost"] = catboost_model
    if tabfm and HAS_TABFM and RUN_TABFM:
        zoo["TabFM"] = lambda: TabFMWrapper("default")
    if tabfm and HAS_TABFM and RUN_TABFM_ENSEMBLE:
        zoo["TabFM ensemble"] = lambda: TabFMWrapper("ensemble")
    return zoo


def run_models(zoo, X_fit, y_fit, X_eval, verbose=True):
    # Fits every model; returns {name: proba on X_eval}, a timing table and {name: fitted model}.
    # A failing model is reported and skipped.
    probas, times, fitted = {}, [], {}
    for name, build in zoo.items():
        t0 = time.time()
        try:
            model = build()
            model.fit(X_fit, y_fit)
            t1 = time.time()
            probas[name] = align_proba(model.classes_, model.predict_proba(X_eval))
            t2 = time.time()
        except Exception:
            print(f"{name:24s} FAILED, skipped:")
            traceback.print_exc()
            continue
        fitted[name] = model
        times.append({"model": name, "fit_s": t1 - t0, "predict_s": t2 - t1})
        if verbose:
            print(f"{name:24s} fit {t1 - t0:7.1f} s | predict {t2 - t1:7.1f} s")
    return probas, pd.DataFrame(times, columns=["model", "fit_s", "predict_s"]).set_index("model"), fitted


class Restrict:
    # Wraps a model so it only sees `cols` (every model gets the same frame; persistence needs the last-class column)
    def __init__(self, model, cols):
        self.model, self.cols = model, cols

    def fit(self, X, y):
        self.model.fit(X[self.cols], y)
        self.classes_ = self.model.classes_
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X[self.cols])


def restricted(zoo, cols):
    return {name: (build if name == "Persistence" else (lambda b=build: Restrict(b(), cols)))
            for name, build in zoo.items()}


def rolling_origin(table, features, last_col, years, zoo, verbose=True):
    # Each year is predicted by models trained only on the years before it; returns the pooled predictions
    cols = list(dict.fromkeys(features + [last_col]))
    zoo = restricted(zoo, features)
    parts, priors, probas, fitted, times, done = [], [], {}, {}, [], []
    for year in years:
        tr, te = table[table["target_year"] < year], table[table["target_year"] == year]
        if te.empty:
            continue
        p, t, models = run_models(zoo, tr[cols], tr["y_class"].values, te[cols], verbose=False)
        for name in p:
            probas.setdefault(name, {})[year] = p[name]
            fitted.setdefault(name, {})[year] = models[name]
        times.append(t)
        parts.append(te)
        priors.append(np.tile(class_prior(tr["y_class"]), (len(te), 1)))
        done.append(year)
        if verbose:
            print(f"{year}: trained on {len(tr):,} rows, predicted {len(te):,} rows with "
                  f"{int(is_onset(te, last_col).sum())} onsets")
    complete = [m for m in probas if list(probas[m]) == done]   # models that ran in every year
    return {"eval": pd.concat(parts), "prior": np.vstack(priors), "cols": cols, "last_col": last_col,
            "probas": {m: np.vstack([probas[m][y] for y in done]) for m in complete},
            "fitted": {m: fitted[m] for m in complete},
            "times": pd.concat(times).groupby(level=0)[["fit_s", "predict_s"]].sum().reindex(complete)}

In [ ]:
def leaderboard(probas, y, last_class, prior, times=None):
    board = pd.DataFrame({m: score(y, p, last_class, prior) for m, p in probas.items()}).T[METRICS].astype(float)
    ranked = board.drop(index="Majority class", errors="ignore")   # never warns, so it is shown but not ranked
    group_ranks = pd.DataFrame({g: pd.concat([ranked[k].rank(ascending=not HIGHER_IS_BETTER[k]) for k in ks], axis=1)
                                .mean(axis=1) for g, ks in RANK_GROUPS.items()})
    board.insert(0, "mean_rank", group_ranks.mean(axis=1))
    if times is not None:
        board = board.join(times)
    return board.sort_values(["mean_rank", "onset_recall"], ascending=[True, False], na_position="last")


def show_board(board, title):
    display(Markdown(f"#### {title}"))
    fmt = {c: "{:.3f}" for c in board.columns}
    fmt.update({c: "{:.0f}" for c in COUNT_METRICS + ["false_alarms_2026"]})
    fmt.update({"mean_rank": "{:.1f}", "fit_s": "{:.1f}", "predict_s": "{:.1f}"})
    styler = board.style.format({c: f for c, f in fmt.items() if c in board.columns}, na_rep="-")
    for col, up in HIGHER_IS_BETTER.items():
        if col in board.columns and col != "onsets" and board[col].notna().any():
            styler = styler.background_gradient(subset=[col], cmap="Greens" if up else "Greens_r")
    display(styler.background_gradient(subset=["mean_rank"], cmap="Blues_r"))
    print("mean_rank = average of the ranks on onset recall, cost (2:1 and 3:1 averaged) and macro-F1, "
          "majority class not ranked (1 = best). Greener = better in each column.")


def is_tabfm(name):
    return name.startswith("TabFM")


def headline_models(board):
    # Persistence, the best-ranked trained model, and every TabFM variant
    trained = [m for m in board.index if m not in BASELINES and not is_tabfm(m)]
    return ["Persistence"] + trained[:1] + [m for m in board.index if is_tabfm(m)]


def color_of(name):
    if is_tabfm(name):
        return "#d62728"
    if name in BASELINES:
        return "#7f7f7f"
    return "#1f77b4"


def metric_bars(board, metrics, title):
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.6 * len(metrics), 0.42 * len(board) + 1.5), sharey=True)
    order = board.sort_values("mean_rank", ascending=False).index
    for ax, m in zip(np.atleast_1d(axes), metrics):
        vals = board.loc[order, m]
        ax.barh(order, vals, color=[color_of(n) for n in order])
        if "Persistence" in board.index:
            ax.axvline(board.loc["Persistence", m], color="k", ls="--", lw=1)
        for i, v in enumerate(vals):
            if pd.notna(v):
                ax.text(v, i, f" {v:.0f}" if m in COUNT_METRICS else f" {v:.3f}", va="center", fontsize=8)
        ax.set_title(m + (" (higher = better)" if HIGHER_IS_BETTER[m] else " (lower = better)"), fontsize=10)
        if HIGHER_IS_BETTER[m] and m not in COUNT_METRICS:
            ax.set_xlim(0, max(1.0, np.nanmax(vals.values) * 1.15))
    fig.suptitle(title + "   (red = TabFM, blue = trained models, grey = baselines, dashed = persistence)", y=1.02)
    plt.tight_layout()
    plt.show()


def confusion_grid(probas, y, prior, title):
    names = list(probas)
    ncol = 4
    nrow = int(np.ceil(len(names) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.6 * nrow))
    for ax, name in zip(axes.ravel(), names):
        cm = confusion_matrix(y, decide(probas[name], prior), labels=CLASSES)
        share = cm / cm.sum(1, keepdims=True).clip(1)
        labels = np.array([[f"{share[i, j]:.0%}\n({cm[i, j]})" for j in range(3)] for i in range(3)])
        sns.heatmap(share, annot=labels, fmt="", cmap="Reds" if is_tabfm(name) else "Blues", vmin=0, vmax=1,
                    cbar=False, ax=ax, xticklabels=["pred 0", "pred 1", "pred 2"], yticklabels=["true 0", "true 1", "true 2"])
        ax.set_title(name, fontsize=10)
    for ax in axes.ravel()[len(names):]:
        ax.axis("off")
    fig.suptitle(title + " · row % (count)", y=1.0)
    plt.tight_layout()
    plt.show()


def bootstrap_report(probas, y, weeks, last_class, prior, title, metrics=None):
    # Intervals per model, then each TabFM variant against the best trained model on the same resamples (paired)
    metrics = metrics or BOOT_METRICS
    names = [m for m in probas if m != "Majority class"]
    boots = week_bootstrap(y, {m: probas[m] for m in names}, weeks, last_class, prior, metrics)
    point = pd.DataFrame({m: score(y, probas[m], last_class, prior, metrics) for m in names}).T.astype(float)
    order = point.sort_values(metrics[0], na_position="first").index
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.6 * len(metrics), 0.4 * len(names) + 1.6), sharey=True)
    for ax, k in zip(np.atleast_1d(axes), metrics):
        lo, hi = boots[k].quantile(0.025), boots[k].quantile(0.975)
        for i, m in enumerate(order):
            ax.plot([lo[m], hi[m]], [i, i], color=color_of(m), lw=2)
            ax.plot(point.loc[m, k], i, "o", color=color_of(m))
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels(order)
        ax.set_title(f"{k}, 95% interval" + ("" if HIGHER_IS_BETTER[k] else " (lower = better)"), fontsize=10)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

    trained = [m for m in names if not is_tabfm(m) and m not in BASELINES]
    rows = []
    for tab in [m for m in names if is_tabfm(m)]:
        for k in metrics:
            col = point.loc[trained, k]
            if col.isna().all():
                continue
            sign = 1 if HIGHER_IS_BETTER[k] else -1
            best = (sign * col).idxmax()
            adv = (sign * (boots[k][tab] - boots[k][best])).dropna()
            rows.append({"TabFM variant": tab, "metric": k, "best trained model": best,
                         "TabFM": point.loc[tab, k], "best trained": point.loc[best, k],
                         "TabFM advantage": sign * (point.loc[tab, k] - point.loc[best, k]),
                         "95% CI low": adv.quantile(0.025), "95% CI high": adv.quantile(0.975),
                         "P(TabFM better)": (adv > 0).mean()})
    if rows:
        table = pd.DataFrame(rows)
        display(Markdown("TabFM advantage is positive when TabFM is better (for costs, that means lower)."))
        display(table.style.format({c: "{:.3f}" for c in table.columns if table[c].dtype.kind == "f"}))
        return boots, table
    print("No TabFM rows to compare (TabFM switched off or failed).")
    return boots, None


def timeline(df, probas, y_col, title, models):
    # Weekly mean predicted P(>= 5k) vs the observed share >= 5k (one row per week on the regional table)
    weekly = pd.DataFrame({"week": df["target_week"].values, "observed": (df[y_col].values > 0).astype(float)})
    for m in models:
        weekly[m] = probas[m][:, 1] + probas[m][:, 2]
    weekly = weekly.groupby("week").mean()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.fill_between(weekly.index, weekly["observed"], step="mid", color="#fdae6b", alpha=0.6, label="observed ≥ 5k")
    for m in models:
        ax.plot(weekly.index, weekly[m], lw=2.2 if is_tabfm(m) else 1.3, label=m)
    ax.set(ylabel="share / mean probability", title=title, ylim=(0, 1))
    ax.legend(fontsize=8, ncol=3)
    plt.tight_layout()
    plt.show()


def onsets_by_year(roll, models):
    # Onsets per forecast year and how many each model caught
    ev = roll["eval"]
    onset = is_onset(ev, roll["last_col"]).values
    years = ev["target_year"].values
    table = {"onsets": pd.Series(onset).groupby(years).sum()}
    for m in models:
        table[m] = pd.Series(onset & (decide(roll["probas"][m], roll["prior"]) > 0)).groupby(years).sum()
    table = pd.DataFrame(table).astype(int)
    table.loc["total"] = table.sum()
    return table


def verdict(board, comparison, label, note=None):
    lines = [f"### {label}", ""]
    if note:
        lines += [note, ""]
    lines.append("| model | onset recall | onsets caught | missed blooms | false alarms | "
                 + " | ".join(f"cost {r}:1" for r in COST_RATIOS) + " | macro-F1 |")
    lines.append("|---" * (6 + len(COST_RATIOS)) + "|")
    for m, r in board.loc[headline_models(board)].iterrows():
        recall = "-" if pd.isna(r["onset_recall"]) else f"{r['onset_recall']:.2f}"
        lines.append(f"| {m} | {recall} | {r['onsets_caught']:.0f} of {r['onsets']:.0f} | {r['missed_blooms']:.0f} | "
                     f"{r['false_alarms']:.0f} | " + " | ".join(f"{r[c]:.0f}" for c in COST_COLS)
                     + f" | {r['macro_f1']:.3f} |")
    lines += ["", f"- Best mean rank: **{board.index[0]}**."]
    if comparison is not None:
        for _, r in comparison.iterrows():
            if r["95% CI low"] > 0:
                call = "clearly better"
            elif r["95% CI high"] < 0:
                call = "clearly worse"
            else:
                call = "not distinguishable"
            lines.append(f"- {r['TabFM variant']} vs {r['best trained model']} on {r['metric']}: TabFM advantage "
                         f"{r['TabFM advantage']:+.3f} [{r['95% CI low']:+.3f}, {r['95% CI high']:+.3f}] → **{call}**.")
    elif not any(is_tabfm(m) for m in board.index):
        lines.append("- TabFM was not run, so there is no TabFM comparison.")
    display(Markdown("\n".join(lines)))

## 3. Regional forecast, validation 2024: train ≤ 2023 → predict 2024
The same split and table as `06_LightGBM_Boosting_Model.ipynb`. 2024
has only **2 onset weeks**, so onset recall here is 0, 0.5 or 1; section 4 is the real onset comparison.

In [ ]:
R_LAST = "last_class"
r_cols = list(dict.fromkeys(R_FEATURES + [R_LAST]))
R_ZOO = model_zoo(R_LAST, LGB_REGIONAL, XGB_DEFAULT)

r_train = regional[regional["split"] == "train"]
r_val = regional[regional["split"] == "validate"]
r_prior = class_prior(r_train["y_class"])
r_val_probas, r_val_times, _ = run_models(restricted(R_ZOO, R_FEATURES), r_train[r_cols], r_train["y_class"].values,
                                          r_val[r_cols])
r_val_board = leaderboard(r_val_probas, r_val["y_class"].values, r_val[R_LAST], r_prior, r_val_times)
show_board(r_val_board, f"Regional · validation 2024 · {len(r_train)} training weeks, {len(r_val)} validation weeks · "
                        f"features: {FEATURE_SET}")
r_val_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_regional_val2024_{FEATURE_SET}.csv"))

## 4. Regional forecast, year by year (main comparison)
Each year in `ROLLING_YEARS` is predicted by models trained only on the years before it (2017 from 2015–2016, …,
2024 from 2015–2023), and the predictions are pooled. Nothing is tuned on these years except the LightGBM settings,
which the LightGBM notebook chose on the training years. Early years have little training data, which is part of the
test: a forecast that only works with many years of history is less useful.

In [ ]:
r_roll = rolling_origin(regional, R_FEATURES, R_LAST, ROLLING_YEARS, R_ZOO)
r_roll_y, r_roll_last = r_roll["eval"]["y_class"].values, r_roll["eval"][R_LAST].values
r_roll_board = leaderboard(r_roll["probas"], r_roll_y, r_roll_last, r_roll["prior"], r_roll["times"])
years_label = f"{ROLLING_YEARS[0]}–{ROLLING_YEARS[-1]}"
show_board(r_roll_board, f"Regional · each year {years_label} predicted from earlier years · {len(r_roll_y)} weeks · "
                         f"features: {FEATURE_SET}")
r_roll_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_regional_rolling_{FEATURE_SET}.csv"))
metric_bars(r_roll_board, ["onset_recall", "false_alarms", COST_COLS[-1], "macro_f1"], f"Regional, {years_label}")

In [ ]:
display(Markdown("**Onsets caught per forecast year**"))
display(onsets_by_year(r_roll, headline_models(r_roll_board)))
confusion_grid(r_roll["probas"], r_roll_y, r_roll["prior"], f"Confusion matrices, regional {years_label}")

### Are the differences real? Week bootstrap
Each dot is a model's score and the bar its 95 % interval when the weeks are resampled. The table compares each TabFM
variant with the best trained model **on the same resamples** (paired), which is tighter than comparing two intervals
by eye. With about 25 onsets, a difference of one or two onsets is noise.

In [ ]:
r_roll_boots, r_roll_vs_best = bootstrap_report(r_roll["probas"], r_roll_y, r_roll["eval"]["week_start"].values,
                                                r_roll_last, r_roll["prior"], f"Regional {years_label}, week bootstrap")
timeline(r_roll["eval"], r_roll["probas"], "y_class", f"Regional forecast week by week, {years_label}",
         headline_models(r_roll_board))

## 5. Regional: environmental feature groups
Each group is added to `regional_core` and scored with the same year-by-year setup. TabFM's pitch is that it finds
interactions without hand-made features, so this checks whether it uses these groups better than the trained models.
Chlorophyll starts in 2018 and gridded wind in 2019, so those columns are empty in the earliest training years. A group
is worth keeping only if it raises onset recall without raising the cost, for more than one model.

In [ ]:
if RUN_REGIONAL_SWEEP:
    sweep_sets = ["core", "core+temperature", "core+chlorophyll", "core+wind", "core+nutrients_bay",
                  "core+rain_optional", "core+all"]
    full_zoo = model_zoo(R_LAST, LGB_REGIONAL, XGB_DEFAULT, tabfm=False)
    sw_zoo = {m: full_zoo[m] for m in ["Logistic regression", "Random forest", "LightGBM"]}
    if HAS_TABFM and RUN_TABFM:
        sw_zoo["TabFM (8 members)"] = lambda: TabFMWrapper("default", n_estimators=8)
    rows = []
    for fs_name in sweep_sets:
        cols = feature_list("regional", fs_name)
        roll = rolling_origin(regional, cols, R_LAST, ROLLING_YEARS, sw_zoo, verbose=False)
        for m, p in roll["probas"].items():
            rows.append({"feature_set": fs_name, "model": m,
                         **score(roll["eval"]["y_class"].values, p, roll["eval"][R_LAST].values, roll["prior"])})
        print(f"{fs_name:20s} done ({len(cols)} features)")
    sweep = pd.DataFrame(rows)
    sweep.to_csv(os.path.join(OUTPUT_DIR, "feature_sweep_regional.csv"), index=False)

    shown = ["onset_recall", COST_COLS[-1], "macro_f1", "auc_any"]
    fig, axes = plt.subplots(1, len(shown), figsize=(5.4 * len(shown), 3.4))
    for ax, metric in zip(axes, shown):
        pivot = sweep.pivot(index="model", columns="feature_set", values=metric)[sweep_sets]
        sns.heatmap(pivot, annot=True, fmt=".0f" if metric in COUNT_METRICS else ".3f",
                    cmap="Greens" if HIGHER_IS_BETTER[metric] else "Greens_r", ax=ax, cbar=False)
        ax.set_title(f"{metric}, regional {years_label}")
        ax.set(xlabel="", ylabel="")
        ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()

    change = sweep.set_index(["model", "feature_set"])[shown]
    change = change.sub(change.xs("core", level="feature_set"), level="model").drop(index="core", level="feature_set")
    display(Markdown(f"**Change from `core` when a group is added** (onset recall and macro-F1: higher is better; "
                     f"{COST_COLS[-1]}: lower is better)"))
    display(change.unstack("model").round(3))

## 6. Regional: which features drive the predictions?
Permutation importance on the section 4 models: shuffle one feature within each forecast year and measure how much the
pooled **AUC ≥5k** drops (5 repeats, no refitting). It works the same way for any model, including TabFM (switch on
`RUN_TABFM_IMPORTANCE`; it needs one TabFM prediction per feature, repeat and year). Importance shows what a model uses,
not what causes blooms.

In [ ]:
def rolling_importance(roll, features, models, n_repeats=5):
    rng = np.random.default_rng(SEED)
    ev = roll["eval"]
    pos = ev["y_class"].values > 0
    out = {}
    for name in models:
        folds = roll["fitted"][name]   # {year: model fitted on the earlier years}

        def pooled_risk(shuffle=None):
            parts = []
            for year, model in folds.items():
                X = ev.loc[ev["target_year"] == year, roll["cols"]].copy()
                if shuffle is not None:
                    X[shuffle] = rng.permutation(X[shuffle].values)
                parts.append(align_proba(model.classes_, model.predict_proba(X))[:, 1:].sum(1))
            return np.concatenate(parts)

        base = roc_auc_score(pos, pooled_risk())
        reps = 2 if is_tabfm(name) else n_repeats
        out[name] = pd.Series({c: np.mean([base - roc_auc_score(pos, pooled_risk(c)) for _ in range(reps)])
                               for c in features})
    return pd.DataFrame(out)


if RUN_REGIONAL_IMPORTANCE:
    t0 = time.time()
    imp_models = [m for m in r_roll["fitted"] if m not in BASELINES and (RUN_TABFM_IMPORTANCE or not is_tabfm(m))]
    imp = rolling_importance(r_roll, R_FEATURES, imp_models)
    imp.to_csv(os.path.join(OUTPUT_DIR, f"permutation_importance_regional_{FEATURE_SET}.csv"))
    print(f"Permutation importance took {time.time() - t0:.0f} s")
    order = imp.mean(axis=1).sort_values().index
    imp.loc[order].plot.barh(figsize=(10, 0.4 * len(order) + 1.5), width=0.8)
    plt.axvline(0, color="k", lw=0.8)
    plt.xlabel(f"drop in pooled AUC ≥5k ({years_label}) when the feature is shuffled")
    plt.title("Permutation importance, regional (bigger = the model relies on it more)")
    plt.tight_layout()
    plt.show()

## 7. Regional final check: train ≤ 2024 → predict 2025, and false alarms in 2026
Every model is refitted on 2015–2024 with the settings above (nothing is tuned on 2025) and predicts 2025. The 2025
regional year has **no onsets** (45 / 1 / 6 weeks of class 0 / 1 / 2), so this checks missed blooms and false alarms
only; onset recall shows as "-". The 2026 holdout has no blooms at all, so it only counts false alarms.

In [ ]:
if SCORE_TEST:
    r_trval = regional[regional["split"].isin(["train", "validate"])]
    r_test = regional[regional["split"] == "test"]
    r_hold = regional[regional["split"] == "holdout"]
    r_prior_tv = class_prior(r_trval["y_class"])
    r_final, r_test_times, _ = run_models(restricted(R_ZOO, R_FEATURES), r_trval[r_cols], r_trval["y_class"].values,
                                          pd.concat([r_test, r_hold])[r_cols])
    r_test_probas = {m: p[:len(r_test)] for m, p in r_final.items()}
    r_test_board = leaderboard(r_test_probas, r_test["y_class"].values, r_test[R_LAST], r_prior_tv, r_test_times)
    r_test_board["false_alarms_2026"] = pd.Series({m: int((decide(p[len(r_test):], r_prior_tv) > 0).sum())
                                                   for m, p in r_final.items()})
    show_board(r_test_board, f"Regional · TEST 2025 · trained on 2015–2024 · features: {FEATURE_SET}")
    r_test_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_regional_test2025_{FEATURE_SET}.csv"))
    print(f"false_alarms_2026 = 2026 holdout weeks (all truly class 0, n = {len(r_hold)}) predicted as 1 or 2")

## 8. Site-week forecast (secondary)
The same comparison per sampling site (`site_week_table_fe.parquet`, 12,379 training rows). Per the advisor, site-level
prediction stays secondary until it is shown to be reliable across sites. This table has many more onsets (90 in 2024,
33 in 2025), so it is also the better place to see whether TabFM's ranking holds with more data.

### Tune LightGBM and XGBoost on the 2024 site-week year
Google's claim is that TabFM matches **tuned** gradient boosting, so the boosted trees get a fair chance: a random
search over the usual settings, scored by 2024 log loss, with early stopping. This makes their 2024 scores slightly
optimistic; the 2025 check below uses no 2025 data for tuning.

In [ ]:
LAST_CLASS = "site_last_class"
train = site[site["split"] == "train"]
val = site[site["split"] == "validate"]
X_tr, y_tr = train[FEATURES], train["y_class"].values
X_va, y_va = val[FEATURES], val["y_class"].values


def random_search(kind, X_tr, y_tr, X_ev, y_ev, n_trials):
    rng = np.random.default_rng(SEED)
    trials = []
    for i in range(n_trials):
        p = dict(learning_rate=float(rng.choice([0.01, 0.02, 0.03, 0.05, 0.1])),
                 subsample=float(rng.choice([0.6, 0.7, 0.8, 0.9, 1.0])),
                 colsample_bytree=float(rng.choice([0.5, 0.7, 0.8, 1.0])),
                 reg_lambda=float(rng.choice([0.0, 1.0, 5.0, 10.0])), n_estimators=3000)
        if kind == "lightgbm":
            p.update(num_leaves=int(rng.choice([4, 7, 15, 31, 63])), subsample_freq=1,
                     min_child_samples=int(rng.choice([10, 20, 50, 100, 200])))
            m = lightgbm_model(p)
            m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)], callbacks=[lgb.early_stopping(100, verbose=False)])
            best_iter = m.best_iteration_
        else:
            p.update(max_depth=int(rng.choice([2, 3, 4, 6, 8])), min_child_weight=float(rng.choice([1, 5, 10, 20])))
            m = xgb.XGBClassifier(objective="multi:softprob", tree_method="hist",
                                  device="cuda" if HAS_GPU else "cpu", random_state=SEED,
                                  early_stopping_rounds=100, eval_metric="mlogloss", **p)
            m.fit(X_tr, y_tr, eval_set=[(X_ev, y_ev)], verbose=False)
            best_iter = m.best_iteration + 1
        loss = log_loss(y_ev, align_proba(m.classes_, m.predict_proba(X_ev)), labels=CLASSES)
        trials.append({**p, "n_estimators": max(int(best_iter), 20), "val_log_loss": loss})
    trials = pd.DataFrame(trials).sort_values("val_log_loss").reset_index(drop=True)
    best = trials.iloc[0].drop("val_log_loss").to_dict()
    int_keys = ["n_estimators", "num_leaves", "min_child_samples", "max_depth", "subsample_freq"]
    best = {k: (int(v) if k in int_keys else float(v)) for k, v in best.items() if pd.notna(v)}
    return best, trials


if TUNE_TREES:
    t0 = time.time()
    LGB_PARAMS, lgb_trials = random_search("lightgbm", X_tr, y_tr, X_va, y_va, N_TRIALS)
    XGB_PARAMS, xgb_trials = random_search("xgboost", X_tr, y_tr, X_va, y_va, N_TRIALS)
    print(f"Tuning took {time.time() - t0:.0f} s")
    print("Best LightGBM:", LGB_PARAMS, "| val log loss", round(lgb_trials['val_log_loss'].iloc[0], 4))
    print("Best XGBoost: ", XGB_PARAMS, "| val log loss", round(xgb_trials['val_log_loss'].iloc[0], 4))
else:
    LGB_PARAMS, XGB_PARAMS = LGB_DEFAULT, XGB_DEFAULT
    print("Tuning off: using the default LightGBM / XGBoost settings")
SITE_LGB = "LightGBM" + (" (tuned)" if TUNE_TREES else "")
SITE_XGB = "XGBoost" + (" (tuned)" if TUNE_TREES else "")

### Site-week validation: train ≤ 2023 → predict 2024

In [ ]:
fit_cols = list(dict.fromkeys(FEATURES + [LAST_CLASS]))
SITE_ZOO = model_zoo(LAST_CLASS, LGB_PARAMS, XGB_PARAMS, lgb_name=SITE_LGB, xgb_name=SITE_XGB)
PRIOR = class_prior(y_tr)
site_val_probas, site_val_times, _ = run_models(restricted(SITE_ZOO, FEATURES), train[fit_cols], y_tr, val[fit_cols])
site_val_board = leaderboard(site_val_probas, y_va, val[LAST_CLASS], PRIOR, site_val_times)
show_board(site_val_board, f"Site-week · validation 2024 · features: {FEATURE_SET} · rule: {DECISION_RULE}")
site_val_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_site_val2024_{FEATURE_SET}.csv"))
metric_bars(site_val_board, ["onset_recall", "false_alarms", COST_COLS[-1], "macro_f1"], "Site-week, validation 2024")

In [ ]:
site_val_boots, site_val_vs_best = bootstrap_report(site_val_probas, y_va, val["week_start"].values,
                                                    val[LAST_CLASS].values, PRIOR, "Site-week 2024, week bootstrap")
timeline(val, site_val_probas, "y_class", "Site-week 2024: mean predicted risk across sites vs share of sites ≥ 5k",
         headline_models(site_val_board))

### Learning curve: does TabFM win with less data?
Each model is trained on a random sample of the site-week training rows (2 seeds) and scored on all of 2024.
Foundation models usually lead at small sizes; the question is whether trained models catch up by the full 12,379
rows. TabFM uses 8 ensemble members here to keep the run short, so its curve is a little below the full model.

In [ ]:
if RUN_LEARNING_CURVE:
    sizes = [250, 500, 1000, 2000, 4000, 8000, len(train)]
    # LightGBM uses the default settings here: the tuned ones (e.g. min_child_samples) assume the full training set
    lc_zoo = {"Logistic regression": logistic, "Random forest": random_forest,
              "LightGBM (default)": lambda: lightgbm_model(LGB_DEFAULT)}
    if HAS_TABFM and RUN_TABFM:
        lc_zoo["TabFM (8 members)"] = lambda: TabFMWrapper("default", n_estimators=8)
    rows = []
    for n in sizes:
        for seed in ([0, 1] if n < len(train) else [0]):
            sub = train.sample(n=n, random_state=seed) if n < len(train) else train
            for name, build in lc_zoo.items():
                try:
                    m = build().fit(sub[FEATURES], sub["y_class"].values)
                    p = align_proba(m.classes_, m.predict_proba(val[FEATURES]))
                    s = score(y_va, p, val[LAST_CLASS], class_prior(sub["y_class"]))
                    rows.append({"n_train": n, "seed": seed, "model": name, **s})
                except Exception as e:
                    print(f"{name} at n={n} failed: {e}")
        print(f"n = {n:>6,} done")
    lc = pd.DataFrame(rows)
    lc.to_csv(os.path.join(OUTPUT_DIR, "learning_curve_site.csv"), index=False)

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
    for ax, metric in zip(axes, ["onset_recall", COST_COLS[-1], "auc_any"]):
        for name, g in lc.groupby("model"):
            g = g.groupby("n_train")[metric].mean()
            ax.plot(g.index, g.values, "o-", lw=2.4 if is_tabfm(name) else 1.4, label=name,
                    color="#d62728" if is_tabfm(name) else None)
        ax.axhline(site_val_board.loc["Persistence", metric], color="k", ls="--", lw=1, label="Persistence")
        ax.set_xscale("log")
        ax.set(xlabel="training rows (log scale)",
               title=f"{metric} on 2024" + ("" if HIGHER_IS_BETTER[metric] else " (lower = better)"))
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

### Site-week final check: train ≤ 2024 → predict 2025, and false alarms in 2026
2025 has 49 at-risk rows, 28 bloom rows and 33 onsets, so use the intervals.

In [ ]:
if SCORE_TEST:
    trval = site[site["split"].isin(["train", "validate"])]
    test = site[site["split"] == "test"]
    holdout = site[site["split"] == "holdout"]
    y_trval, y_te = trval["y_class"].values, test["y_class"].values
    prior_tv = class_prior(y_trval)

    # Boosted trees keep their tuned settings, with ~20 % more trees for the extra year of data
    lgb_final = dict(LGB_PARAMS, n_estimators=int(LGB_PARAMS["n_estimators"] * 1.2))
    xgb_final = dict(XGB_PARAMS, n_estimators=int(XGB_PARAMS["n_estimators"] * 1.2))
    final_zoo = model_zoo(LAST_CLASS, lgb_final, xgb_final, lgb_name=SITE_LGB, xgb_name=SITE_XGB)
    site_final, site_test_times, _ = run_models(restricted(final_zoo, FEATURES), trval[fit_cols], y_trval,
                                                pd.concat([test, holdout])[fit_cols])
    site_test_probas = {m: p[:len(test)] for m, p in site_final.items()}
    site_test_board = leaderboard(site_test_probas, y_te, test[LAST_CLASS], prior_tv, site_test_times)
    site_test_board["false_alarms_2026"] = pd.Series({m: int((decide(p[len(test):], prior_tv) > 0).sum())
                                                      for m, p in site_final.items()})
    show_board(site_test_board, f"Site-week · TEST 2025 · trained on 2015–2024 · features: {FEATURE_SET}")
    site_test_board.to_csv(os.path.join(OUTPUT_DIR, f"leaderboard_site_test2025_{FEATURE_SET}.csv"))
    print(f"false_alarms_2026 = 2026 holdout rows (all truly class 0, n = {len(holdout)}) predicted as 1 or 2")
    site_test_boots, site_test_vs_best = bootstrap_report(site_test_probas, y_te, test["week_start"].values,
                                                          test[LAST_CLASS].values, prior_tv,
                                                          "Site-week 2025, week bootstrap")

## 9. Verdict
Printed from the tables above. "Clearly better" means the paired 95 % interval of the TabFM advantage excludes zero.
Each table lists persistence, the best-ranked trained model and every TabFM variant.

In [ ]:
verdict(r_roll_board, r_roll_vs_best, f"Main result: regional forecast, {years_label} predicted year by year",
        note=f"Pooled over {len(r_roll_y)} weeks. This is the comparison the advisor's priorities point to.")
verdict(r_val_board, None, "Regional, validation 2024",
        note="Only 2 onsets, so no intervals.")
if SCORE_TEST:
    verdict(r_test_board, None, "Regional, test 2025",
            note="No onsets in 2025, so this checks missed blooms and false alarms only.")
verdict(site_val_board, site_val_vs_best, "Site-week (secondary), validation 2024")
if SCORE_TEST:
    verdict(site_test_board, site_test_vs_best, "Site-week (secondary), test 2025")

### How to read the result
- **TabFM clearly better on the regional year-by-year comparison, on onset recall without a higher cost:** worth
  carrying forward as a candidate. Keep a tree model or logistic regression next to it for the driver analysis,
  because TabFM has no coefficients, splits or SHAP values, only permutation importance.
- **Not distinguishable:** prefer the trained model. It explains its drivers, runs in seconds without a GPU, and has
  no licence restriction. The default TabFM weights are licensed for **non-commercial use only**, so Biointerphase
  could not use them in a product.
- **Clearly worse:** report it as a tested and rejected approach. That is still a useful result for the write-up.

In every case, the gap to **persistence** on onset recall and cost matters more than small gaps between models. Most
of the skill comes from bloom history, so the data limits the forecast more than the choice of learner does.